# Federalist Papers Experiment: Split Before Feature Extraction

In [1]:
# Step 1: Imports and Paths
from pathlib import Path
import pandas as pd
from lexos.dtm import DTM, Vectorizer
from lexos.classification import trainer
from lexos.corpus.corpus_stats import CorpusStats
from lexos.scrubber.scrubber import Scrubber
from lexos.tokenizer import Tokenizer
from sklearn.model_selection import train_test_split

BASE = Path.cwd()
DATA = BASE / "fed_papers"

print("Base:", BASE)
print("Data dir exists:", DATA.exists())

Base: c:\Users\gabal\LocalFiles\Lexos_Independant_Research\lexos\doc_src\docs\tutorials\classification
Data dir exists: True


In [2]:
# Step 2: Collect and Split File Paths
train_dirs = ["HAMILTON", "MADISON"]
test_dirs = ["COAUTHORED", "DISPUTED"]

# Collect all file paths and labels
data_files = []
data_labels = []

for d in train_dirs:
    files = sorted((DATA / d).glob("*.txt"))
    data_files.extend(files)
    data_labels.extend([d] * len(files))

for d in test_dirs:
    files = sorted((DATA / d).glob("*.txt"))
    data_files.extend(files)
    data_labels.extend([d] * len(files))

# Split into training and testing sets
train_files, test_files, train_labels, test_labels = train_test_split(
    data_files, data_labels, test_size=0.3, random_state=42, stratify=data_labels
)

print("Train files:", len(train_files), "Test files:", len(test_files))

Train files: 56 Test files: 24


In [3]:
# Step 3: Scrubber and Tokenizer
scrubber = Scrubber()
scrubber.add_pipe("lower_case")
scrubber.add_pipe("digits")
scrubber.add_pipe("punctuation")

tokenizer = Tokenizer(model="en_core_web_sm")

In [5]:
# Fit CorpusStats and Vectorizer on training data
stats_train = CorpusStats(docs=train_docs_for_corpus_stats, raw_texts=raw_texts_train)
stats_df_train = stats_train.doc_stats_df.copy()

# Initialize and fit the DTM vectorizer
dtm_train = DTM(vectorizer=Vectorizer(max_features=500))
dtm_train(docs=train_docs_for_dtm, labels=train_labels)

print("Training data processed.")

Training data processed.


In [6]:
# Step 5: Process Testing Files

test_docs_for_dtm = []
raw_texts_test = []

for f in test_files:
    raw = f.read_text(encoding="utf-8")
    clean = scrubber.scrub(raw)
    spacy_doc = tokenizer(clean)
    tokens = [t.text for t in spacy_doc if t.text.strip()]
    test_docs_for_dtm.append(tokens)
    raw_texts_test.append(raw)

# Transform test data using the fitted vectorizer
stats_test = CorpusStats(
    docs=[(f.name, f.name, tokens) for f, tokens in zip(test_files, test_docs_for_dtm)],
    raw_texts=raw_texts_test,
)
stats_df_test = stats_test.doc_stats_df.copy()
dtm_test = vectorizer.transform(test_docs_for_dtm)

print("Testing data processed.")

AttributeError: 'Vectorizer' object has no attribute 'transform'

In [ ]:
# Step 6: Combine Features
import scipy.sparse

# Combine CorpusStats and DTM features for training data
dtm_df_train = pd.DataFrame(
    vectorizer.transform(train_docs_for_dtm).toarray(),
    columns=vectorizer.sorted_terms_list,
)
combined_df_train = pd.concat(
    [stats_df_train.reset_index(drop=True), dtm_df_train.reset_index(drop=True)], axis=1
)

# Combine CorpusStats and DTM features for testing data
dtm_df_test = pd.DataFrame(dtm_test.toarray(), columns=vectorizer.sorted_terms_list)
combined_df_test = pd.concat(
    [stats_df_test.reset_index(drop=True), dtm_df_test.reset_index(drop=True)], axis=1
)

print("Features combined.")

In [ ]:
# Step 7: Train Classifier
feature_cols = list(stats_df_train.columns[1:]) + vectorizer.sorted_terms_list

X_train = combined_df_train[feature_cols].values
y_train = train_labels

X_test = combined_df_test[feature_cols].values
y_test = test_labels

clf = trainer.fit_classifier(
    feature_matrix=X_train,
    target_labels=y_train,
    model="decision_tree",
    random_state=42,
    normalize="standard",
)

print("Classifier trained.")

In [ ]:
# Step 8: Evaluate Classifier
from sklearn.metrics import classification_report

y_pred_test = clf.predict(X_test)
print("Classification Report:")
print(classification_report(y_test, y_pred_test))